# Transform Addresses Data
1. Create one record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address
2. Write trasnformed data to the Silver schema

In [0]:
df_addresses = spark.table('gizmobox.bronze.py_addresses')
display(df_addresses)

## 1. Create one record for each customer with both addresses, one for each address_type

[Documentation pivot](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.pivot.html)

In [0]:
from pyspark.sql import functions as F

df_pivoted_addresses = (
    df_addresses
    .groupBy('customer_id')
    .pivot('address_type', ['billing', 'shipping'])
    .agg(
        F.max("address_line_1").alias('address_line_1'),
        F.max("city").alias('city'),
        F.max("state").alias('state'),
        F.max("postcode").alias('postcode')
    )
)

display(df_pivoted_addresses)

## 2. Write transformed data to the Silver Schema

In [0]:
df_pivoted_addresses.writeTo('gizmobox.silver.py_addresses').createOrReplace()

In [0]:
%sql
SELECT * FROM gizmobox.silver.py_addresses